# Inspect a saved World

Load any `world.json` artifact and walk every inspection surface.

**Before you click "Run All":** `load_or_build_world` shows an interactive
consent prompt on a cache miss and raises `LLMBuildAbortedError` in non-TTY
contexts (CI, scheduled jobs). This notebook targets an *already-saved* world
so it never triggers an LLM call — but be careful if you point it at an
unsaved world name.

The demo below uses the offline fixture produced by `example_llm_world_offline.py`
(6-item fashion catalog, no OpenAI key required).


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from dataclasses import replace
from pathlib import Path

sys.path.append('../')

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

from src.llm.world_builder import World


## Load a saved world

Point `WORLD_PATH` at any `world.json` produced by `WorldBuilder.build()` +
`World.to_json()` (notebook 01) or the committed offline fixture at
`notebooks/fixtures/world_fashion_retail_offline.json` (6-item fashion catalog,
no OpenAI key required).


In [ ]:
WORLD_PATH = Path('../notebooks/fixtures/world_fashion_retail_offline.json')

world = World.from_json(WORLD_PATH)
print(f'Loaded world: {len(world.catalog)} catalog items, '
      f'{len(world.store_templates)} store templates')


### Meta — provenance

Records who built this world: archetype, item count, model name, builder
version, and build timestamp. Use this to audit which LLM run produced a
given artifact.


In [ ]:
world.meta_df()


### Market — demand and seasonality parameters

Domain-meaningful fields (`cycle_len`, `peak_factor`, `off_factor`,
`init_demand`, `price_elasticity`, `regions`, `season_months`) are LLM-authored;
math defaults (`sigma`, `crn_draws`, etc.) are merged in by `WorldBuilder`.


In [ ]:
world.market_df()


### Store templates — retail footprint

One row per template. Templates are reusable profiles: `Scenario.from_world`
instantiates concrete `StoreInstance` objects from them via `make_stores`.
`capacity`, `init_balance`, and similar fields can hold `Distribution` objects
when constructed programmatically; here they are floats from the JSON artifact.


In [ ]:
world.store_templates_df().set_index('id')


### Catalog — items for sale

One row per SKU. `margin = base_price - unit_cost` is derived. `freshness_alpha`
and `freshness_decay` govern the hype-curve multiplier applied at the store level.
`stage_change_probs` controls lifecycle-stage transition rates; `None` means
the simulation falls back to `ItemLifecycleParams` defaults.


In [ ]:
world.catalog_df()


## (b) Edit a single store template

Use `dataclasses.replace` to produce a new `World` with one template mutated.
Save it under a distinct name so the original artifact is preserved.


In [ ]:
# Bump flagship capacity from 500 → 800 units, save under a new name.
world_b = replace(
    world,
    store_templates={
        **world.store_templates,
        'flagship': replace(world.store_templates['flagship'], capacity=800),
    },
)

world_b.to_json('../data/worlds/fashion_retail_offline_b/world.json')
print('flagship capacity (original):', world.store_templates['flagship'].capacity)
print('flagship capacity (edited):  ', world_b.store_templates['flagship'].capacity)
print('Saved to data/worlds/fashion_retail_offline_b/world.json')


## (c) Replace store templates wholesale

Swap the entire `store_templates` dict — useful when migrating a world to a
different retail footprint (e.g. adding a 'kiosk' template or removing a region).


In [ ]:
from dataclasses import fields as dc_fields

# Clone 'standard' as a new 'kiosk' template with reduced capacity and balance.
standard = world.store_templates['standard']
kiosk = replace(standard, id='kiosk', capacity=80.0, init_balance=4_000.0,
                init_active_count=2)

world_c = replace(
    world,
    store_templates={'flagship': world.store_templates['flagship'], 'kiosk': kiosk},
)

world_c.to_json('../data/worlds/fashion_retail_offline_c/world.json')
print('New template keys:', list(world_c.store_templates.keys()))
print('Saved to data/worlds/fashion_retail_offline_c/world.json')
world_c.store_templates_df().set_index('id')


## Re-run behaviour

All three `world.to_json(...)` calls above overwrite their target files on
re-run (idempotent). Demo artifacts (`fashion_retail_offline_b/`,
`fashion_retail_offline_c/`) are written to `data/worlds/` which is
gitignored, so they will not appear in commits.
